# 01 - Retention and engagement KPIs (module A)

**Question.** How many players come back, and how does coming back relate to how much they play?

The same queries are in `sql/01_retention.sql` (with comments). `retention_1` / `retention_7` are booleans:
casting them to `INT` gives 0 or 1, so `AVG()` is the share of players who came back.

In [1]:
import duckdb

# Read-only: this notebook only queries the data.
con = duckdb.connect("../game.duckdb", read_only=True)

## 1. Global KPIs

In [2]:
con.sql("""
    SELECT
      COUNT(*)                              AS players,
      ROUND(100 * AVG(retention_1::INT), 2) AS retention_d1_pct,
      ROUND(100 * AVG(retention_7::INT), 2) AS retention_d7_pct
    FROM cookie_cats
""").show()

┌─────────┬──────────────────┬──────────────────┐
│ players │ retention_d1_pct │ retention_d7_pct │
│  int64  │      double      │      double      │
├─────────┼──────────────────┼──────────────────┤
│   90189 │            44.52 │            18.61 │
└─────────┴──────────────────┴──────────────────┘



## 2. Retention by game version

Description only: the statistical test is in notebook 02.

In [3]:
con.sql("""
    SELECT
      version,
      COUNT(*)                              AS players,
      ROUND(100 * AVG(retention_1::INT), 2) AS retention_d1_pct,
      ROUND(100 * AVG(retention_7::INT), 2) AS retention_d7_pct
    FROM cookie_cats
    GROUP BY version
    ORDER BY version
""").show()

┌─────────┬─────────┬──────────────────┬──────────────────┐
│ version │ players │ retention_d1_pct │ retention_d7_pct │
│ varchar │  int64  │      double      │      double      │
├─────────┼─────────┼──────────────────┼──────────────────┤
│ gate_30 │   44700 │            44.82 │            19.02 │
│ gate_40 │   45489 │            44.23 │             18.2 │
└─────────┴─────────┴──────────────────┴──────────────────┘



## 3. Engagement funnel: day-7 retention by rounds played

`CASE WHEN` builds the bands. Players with 0 rounds installed the game but never played.
Retention rises sharply with engagement, but this is a correlation: playing a lot is partly a
consequence of staying.

In [4]:
con.sql("""
    SELECT
      CASE
        WHEN sum_gamerounds = 0   THEN '0 rounds'
        WHEN sum_gamerounds <= 5  THEN '1-5'
        WHEN sum_gamerounds <= 20 THEN '6-20'
        WHEN sum_gamerounds <= 50 THEN '21-50'
        ELSE '50+'
      END AS band,
      COUNT(*)                              AS players,
      ROUND(100 * AVG(retention_7::INT), 2) AS retention_d7_pct
    FROM cookie_cats
    GROUP BY band
    ORDER BY MIN(sum_gamerounds)
""").show()

┌──────────┬─────────┬──────────────────┐
│   band   │ players │ retention_d7_pct │
│ varchar  │  int64  │      double      │
├──────────┼─────────┼──────────────────┤
│ 0 rounds │    3994 │             0.73 │
│ 1-5      │   20723 │             1.37 │
│ 6-20     │   25085 │             4.89 │
│ 21-50    │   17575 │            15.34 │
│ 50+      │   22812 │             55.0 │
└──────────┴─────────┴──────────────────┘



## 4. Engagement quartiles (window function)

`NTILE(4) OVER (ORDER BY ...)` cuts the players into 4 groups of equal size without collapsing the rows.
Players with the same number of rounds can fall in two quartiles, so bounds can overlap by one value.

In [5]:
con.sql("""
    WITH quartiles AS (
      SELECT *, NTILE(4) OVER (ORDER BY sum_gamerounds) AS quartile
      FROM cookie_cats
    )
    SELECT
      quartile,
      COUNT(*)            AS players,
      MIN(sum_gamerounds) AS min_rounds,
      MAX(sum_gamerounds) AS max_rounds,
      ROUND(100 * AVG(retention_7::INT), 2) AS retention_d7_pct
    FROM quartiles
    GROUP BY quartile
    ORDER BY quartile
""").show()

┌──────────┬─────────┬────────────┬────────────┬──────────────────┐
│ quartile │ players │ min_rounds │ max_rounds │ retention_d7_pct │
│  int64   │  int64  │   int64    │   int64    │      double      │
├──────────┼─────────┼────────────┼────────────┼──────────────────┤
│        1 │   22548 │          0 │          5 │             1.22 │
│        2 │   22547 │          5 │         16 │              3.9 │
│        3 │   22547 │         16 │         51 │            13.96 │
│        4 │   22547 │         51 │      49854 │            55.35 │
└──────────┴─────────┴────────────┴────────────┴──────────────────┘



## 5. Extreme values

One player has 49,854 rounds, about 17 times the next one. Cause unknown (error, bot or real player).
**Decision:** keep this player in retention rates (1 out of 90,189 changes nothing) and exclude it from any
analysis based on the number of rounds.

In [6]:
con.sql("""
    SELECT userid, version, sum_gamerounds
    FROM cookie_cats
    ORDER BY sum_gamerounds DESC
    LIMIT 10
""").show()

┌─────────┬─────────┬────────────────┐
│ userid  │ version │ sum_gamerounds │
│  int64  │ varchar │     int64      │
├─────────┼─────────┼────────────────┤
│ 6390605 │ gate_30 │          49854 │
│  871500 │ gate_30 │           2961 │
│ 3271615 │ gate_40 │           2640 │
│ 4832608 │ gate_30 │           2438 │
│ 5346171 │ gate_40 │           2294 │
│ 5133952 │ gate_30 │           2251 │
│ 9640085 │ gate_30 │           2156 │
│ 4090246 │ gate_40 │           2124 │
│ 9791599 │ gate_40 │           2063 │
│  725080 │ gate_40 │           2015 │
└─────────┴─────────┴────────────────┘
  10 rows                  3 columns



In [7]:
con.close()